In [54]:
import sys
import os
import json

# go one level up from notebooks/ so the next imports work
project_root = os.path.abspath("..")
sys.path.append(project_root)

In [55]:
from app.ingestion.reader import read_docs
from app.ingestion.embedding import get_embeddings, text_embedding
from app.clients import get_db_connection, get_mistral_client

In [56]:
configs = [
    {"id": "config_1", "chunk_size": 800, "temperature": 0.1, "top_k": 5},
    {"id": "config_2", "chunk_size": 250, "temperature": 0.2, "top_k": 9},
    {"id": "config_3", "chunk_size": 600, "temperature": 0.8, "top_k": 6},
    {"id": "config_4", "chunk_size": 600, "temperature": 0.0, "top_k": 3},
    {"id": "config_5", "chunk_size": 1000, "temperature": 0.4, "top_k": 1},
]

In [57]:
questions = [
    "How can I create an API token?",
    "How can I trigger a rescan of a project?",
    "How can I upload an SBOM manually?",
    "How can i add a git reference to my attestation?",
    "How can I migrate from manual SBOM uploads to automated scanning?"
]

In [65]:
# split recursively for a hierarchy of separators
# attempt to split on high-level separators first, then move to increasingly finer separators if chunks remain too large
def recursive_chunking(docs: str, chunk_size : int, separators: list[str] = ["\n\n", "\n", ". ", " "]):
    # base case
    if len(docs) <= chunk_size:
        return [docs]

    sep : str = separators[0]
    parts : list[str] = docs.split(sep)

    chunks : list[str] = []
    current_chunk : str = ""

    for part in parts:
        # skip empty parts
        if not part.strip():
            continue

        piece : str = part + sep

        # accumulate until chunk would exceed size
        if len(current_chunk) + len(piece) <= chunk_size:
            current_chunk += piece
        else:
            # finalize current chunk
            if len(current_chunk) > chunk_size and len(separators) > 1:
                chunks.extend(
                    recursive_chunking(current_chunk, chunk_size, separators[1:])
                )
            else:
                chunks.append(current_chunk.strip())
            # start new chunk
            current_chunk = piece  

    # append remaining chunk
    if current_chunk:
        if len(current_chunk) > chunk_size and len(separators) > 1:
            chunks.extend(
                recursive_chunking(current_chunk, chunk_size, separators[1:])
            )
        else:
            chunks.append(current_chunk.strip())

    return chunks


In [59]:
def retrieve_top_k(query: str, k: int = 5):
    embedding = text_embedding(query)

    conn = get_db_connection()
    cur = conn.cursor()

    cur.execute("""
        SELECT content,
               1 - (embedding <=> %s::vector) AS similarity
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (embedding, embedding, k))

    results = cur.fetchall()
    cur.close()
    conn.close()
    return results


In [60]:
client = get_mistral_client()

def generate_response(query: str, temp: float, context: list[tuple[str, float]]) -> str:
    # format context
    MAX_CONTEXT_CHARS = 3000
    context_text = ""
    for content, _ in context:
        if len(context_text) + len(content) > MAX_CONTEXT_CHARS:
            break
        context_text += "\n\n" + content

    prompt = f"""
    You are the DevGuard Documentation Assistant.
    Your primary role is to provide accurate, context-aware technical assistance while maintaining a professional and helpful tone. 
    If context is unavailable but the user request is relevant: State: "I couldn't find specific sources on DevGuard docs, but here's my understanding: [Your Answer]."
    If the user's request is not relevant to DevGuard at all, please refuse user's request and reply something like: "Sorry, I couldn't help with that. However, if you have any questions related to DevGuard, I'd be happy to assist!" 
    Please generate your response using appropriate Markdown formats, including bullets and bold text, to make it reader friendly.
    If the answer cannot be answered using the context, say you don't know.
    Context:
    {context_text}

    Question:
    {query}
    """

    message= [{"role": "user", "content": prompt}]

    """
        Toggling the safe prompt will prepend your messages with the following system prompt:
        Always assist with care, respect, and truth. Respond with utmost utility yet securely. Avoid harmful, unethical, prejudiced, or negative content. Ensure replies promote fairness and positivity.
    """
    response = client.chat.complete(
        model="mistral-medium-2508",
        messages=message,
        safe_prompt=True,
        temperature=temp
    )
    return str(response.choices[0].message.content)


In [ ]:
# create chunks and embeddings once per chunk_size, store in DB
def prepare_embeddings(docs: str, chunk_size: int) -> list[str]:
    chunks = recursive_chunking(docs, chunk_size)
    embeddings = get_embeddings(chunks)

    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("TRUNCATE documents;")  # clear old embeddings
    for chunk, emb in zip(chunks, embeddings):
        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
            (chunk, emb)
        )
    conn.commit()
    cur.close()
    conn.close()
    print(f"{len(chunks)} chunks with embeddings stored in DB.")
    return chunks

In [62]:
def answer_questions(temp: float, top_k: int):
    results = []
    for question in questions:
        context = retrieve_top_k(question, top_k)
        response = generate_response(question, temp, context)
        results.append({"question": question, "answer": response})
        print("Answered question:", question)
    return results


In [ ]:
def run_experiments():
    docs = read_docs()  # only once
    data = []

    # loop over chunk sizes
    unique_chunk_sizes = sorted(set(config["chunk_size"] for config in configs))
    for chunk_size in unique_chunk_sizes:
        print("Preparing embeddings for chunk size:", chunk_size)
        prepare_embeddings(docs, chunk_size)

        # loop over remaining params
        for config in [c for c in configs if c["chunk_size"] == chunk_size]:
            print("Generating responses for config:", config["id"])
            results = answer_questions(config["temperature"], config["top_k"])
            data.append({
                "config": config["id"],
                "chunk_size": chunk_size,
                "temperature": config["temperature"],
                "top_k": config["top_k"],
                "results": results
            })
    return data

In [ ]:
# run pipeline and save results
data = run_experiments()
with open("data.json", "w") as f:
    json.dump(data, f, indent=2)
print("All done! Results saved in sample.json")


Preparing embeddings for chunk size: 250
596 chunks with embeddings stored in DB.
Generating responses for config: config_2
Answered question: How can I create an API token?
Answered question: How can I trigger a rescan of a project?
Answered question: How can I upload an SBOM manually?
Answered question: How can i add a git reference to my attestation?
Answered question: How can I migrate from manual SBOM uploads to automated scanning?
Preparing embeddings for chunk size: 600
223 chunks with embeddings stored in DB.
Generating responses for config: config_3
Answered question: How can I create an API token?
Answered question: How can I trigger a rescan of a project?
Answered question: How can I upload an SBOM manually?
Answered question: How can i add a git reference to my attestation?
Answered question: How can I migrate from manual SBOM uploads to automated scanning?
Generating responses for config: config_4
Answered question: How can I create an API token?
Answered question: How can